# Day 3 - Conversational AI - aka Chatbot!


In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [3]:
# Initialize

openai = OpenAI()
MODEL = "gpt-4.1-mini"

In [4]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [ ]:
def chat(message, history):
    return "bananas"


# this type of callback function is called a one way conversation - when user types in the chat, we say the computer should always return 'bananas'
# when we type something and press "enter" -> the callback function 'chat' is called.

In [6]:
gr.ChatInterface(fn=chat, type="messages").launch()
# to make an instant message app UI, just specify the type as "messages"

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [7]:
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()
# gradio will help us to automatically pass in the current message and history? we don't need to configure this? - YES! (the contents of the user interface is passed in as 'history')
# this 'history' is the openAI 'messages' format.

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## OK! Let's write a slightly better chat callback!


In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # this is a list comprehension that clears out the 'metadata' field in the dicts, this is CRUCIAL for gemini model as it does not like extra fields being passed in to its messages and WILL GET ERROR, BUT openAI does not care, so even if you have metadata inside it's ok

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    # we are just adding the lists together to create one big list.

    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [11]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [12]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response
        # gradio will see that this function is a generator and will automatically keep calling the function(iteration) until all the results have been returned and shown.

In [13]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again


In [16]:
system_message = (
    "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."
)

# in the prompt, we have provided an example like "if the customer says then you should reply..." -> this is an example of one-shot prompting. (this pushes the model to replicate that example.)

In [17]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [19]:
system_message += (
    "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"
)

# this is multi-shot prompting - giving it different examples of how to handle different situations.

In [20]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    relevant_system_message = system_message

    # 'hacky' trick: you can customise the system prompt BASED ON WHAT THE USER SAYS! OMG!
    # sneak peak at RAG -> adding in more context if we need to. RAG is a bit more sophisticated and bullet proof, because if the user used a synonym of belt then it would not work.
    # why not just add this prompt directly in system prompt? wif we have a large store with 1000 different products, this prompt would become so BIG and model's accuracy will start to degrade, tokens and cost will increase.
    # RAG is an inference time technique (not about training the model, it's the SAME model), a clever way to insert more info into the prompt and provide the model more expertise.
    if "belt" in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    messages = (
        [{"role": "system", "content": relevant_system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [23]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>
